# PrefVLM-MVP — Qwen VLM Inference via vLLM (Colab)

**Uses vLLM for fast batched inference — ~5-10× faster than HuggingFace generate().**

**Steps:**
1. Run Cell 1 (install — takes ~3 min)
2. Mount Google Drive or upload `qwen_batch.jsonl` from `data/`
3. Run all remaining cells
4. Download `qwen_results.jsonl` and place it in `data/`
5. Back on your machine run: `uv run python -m prefvlm.run_mvp --stage qwen-ingest`

**Runtime:** A100 recommended (free Colab T4 works but is slower). vLLM requires CUDA 11.8+.

In [ ]:
# Cell 1 — Install vLLM + dependencies
# vLLM installs its own compatible transformers version
!pip install -q vllm qwen-vl-utils pillow
print('Install complete.')

In [ ]:
# Cell 2 — Mount Drive (preferred) or upload file
import os

USE_DRIVE = True  # Set False to use file upload widget instead

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust this path to where you copied qwen_batch.jsonl in Drive
    batch_path = '/content/drive/MyDrive/prefvlm/qwen_batch.jsonl'
    print(f'Using Drive path: {batch_path}')
else:
    from google.colab import files
    print('Upload qwen_batch.jsonl from your data/ directory')
    uploaded = files.upload()
    batch_path = list(uploaded.keys())[0]
    print(f'Uploaded: {batch_path}')

assert os.path.exists(batch_path), f'File not found: {batch_path}'

In [ ]:
# Cell 3 — Load and inspect batch
import json
from collections import Counter

with open(batch_path) as f:
    items = [json.loads(l) for l in f if l.strip()]

print(f'Total items: {len(items)}')
print('Condition breakdown:', Counter(i['condition'] for i in items))
print('First item keys:', list(items[0].keys()))

In [ ]:
# Cell 4 — Load model with vLLM
import sys
import torch
from vllm import LLM, SamplingParams
from transformers import AutoProcessor

# Model selection: Qwen2.5-VL-7B has stable vLLM support
MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'
print(f'Using {MODEL_ID}')

# ── vLLM v0.20 + Colab workaround ──────────────────────────────────────────
# vLLM's EngineCore subprocess calls sys.stdout.fileno() inside suppress_stdout().
# Colab replaces sys.stdout with IPython's OutStream which has no real fd →
# raises io.UnsupportedOperation: fileno.
# Fix: temporarily restore sys.__stdout__ for the duration of LLM.__init__.
# ───────────────────────────────────────────────────────────────────────────
print(f'Loading {MODEL_ID} with vLLM ...')
_colab_stdout = sys.stdout
sys.stdout = sys.__stdout__
try:
    llm = LLM(
        model=MODEL_ID,
        dtype='bfloat16',
        max_model_len=8192,
        limit_mm_per_prompt={'image': 1},
        gpu_memory_utilization=0.90,
        enforce_eager=False,  # set True if you hit CUDA graph errors on T4
    )
finally:
    sys.stdout = _colab_stdout  # always restore Colab output

processor = AutoProcessor.from_pretrained(MODEL_ID)
print('Model loaded and ready.')

In [ ]:
# Cell 5 — Prepare all inputs as a single vLLM batch
import base64, io
from PIL import Image
from tqdm.notebook import tqdm

MAX_NEW_TOKENS = 1024
TEMPERATURE    = 0.7
CHUNK_SIZE     = 150  # items per vLLM generate() call — tune down if OOM

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    max_tokens=MAX_NEW_TOKENS,
)

print(f'Preparing {len(items)} prompts + images...')
vllm_inputs = []

for item in tqdm(items, desc='Building inputs'):
    # Decode image
    img_bytes = base64.b64decode(item['image_b64'])
    pil_img = Image.open(io.BytesIO(img_bytes)).convert('RGB')

    # Build message list
    messages = []
    if item.get('system_text'):
        messages.append({'role': 'system', 'content': item['system_text']})
    messages.append({
        'role': 'user',
        'content': [
            {'type': 'image', 'image': pil_img},
            {'type': 'text',  'text':  item['user_text']},
        ]
    })

    # Apply chat template (produces tokenized-prompt string with image placeholders)
    prompt_str = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    vllm_inputs.append({
        'prompt':           prompt_str,
        'multi_modal_data': {'image': pil_img},
    })

print(f'Inputs ready: {len(vllm_inputs)} items')

In [ ]:
# Cell 6 — Run vLLM batch inference
import math

results      = []
n_chunks     = math.ceil(len(vllm_inputs) / CHUNK_SIZE)
all_outputs  = []

print(f'Running vLLM inference: {len(vllm_inputs)} items in {n_chunks} chunk(s) of {CHUNK_SIZE}')

for chunk_idx in range(n_chunks):
    start = chunk_idx * CHUNK_SIZE
    end   = min(start + CHUNK_SIZE, len(vllm_inputs))
    chunk = vllm_inputs[start:end]
    print(f'  Chunk {chunk_idx+1}/{n_chunks}: items {start}–{end-1}')
    chunk_outputs = llm.generate(chunk, sampling_params)
    all_outputs.extend(chunk_outputs)

# Collect
for item, output in zip(items, all_outputs):
    response_text = output.outputs[0].text.strip()
    results.append({
        'scenario_id':  item['scenario_id'],
        'question_id':  item['question_id'],
        'persona_id':   item['persona_id'],
        'persona_name': item['persona_name'],
        'condition':    item['condition'],
        'model':        MODEL_ID,
        'response':     response_text,
    })

print(f'\nDone. {len(results)} responses generated.')
# Quick sanity check
r = results[0]
print(f"\nSample [{r['scenario_id']} | {r['condition']} | {r['persona_name']}]:")
print(r['response'][:300])

In [ ]:
# Cell 7 — Save to Drive and download
import json, os
from google.colab import files

# Save to Drive (persistent backup)
drive_out = '/content/drive/MyDrive/prefvlm/qwen_results.jsonl'
os.makedirs(os.path.dirname(drive_out), exist_ok=True)
with open(drive_out, 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')
print(f'Saved {len(results)} results to Drive: {drive_out}')

# Also save locally and trigger download
local_out = 'qwen_results.jsonl'
with open(local_out, 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')

files.download(local_out)
print('Download started — place qwen_results.jsonl in your data/ directory.')
print('Then run: uv run python -m prefvlm.run_mvp --stage qwen-ingest')